# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [ ]:
%pip install --quiet datasets evaluate transformers[sentencepiece]


Note: you may need to restart the kernel to use updated packages.


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [ ]:
# pandas handles the raw tabular data; `Dataset` is Hugging Face's in-memory
# table format that integrates cleanly with tokenizers and the Trainer API.
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [ ]:
# Load and inspect the SMS Spam dataset.
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)        # read the parquet file straight into a pandas DataFrame
hf_dataset = Dataset.from_pandas(df)   # wrap the DataFrame as a Hugging Face Dataset

# Index ranges that slice the dataset into a 4,000-row train split and a
# 1,000-row validation split (rows 4,000–4,999).
TRAIN_START = 0
TRAIN_END = 4000   # first 4,000 samples are used for training
VAL_START = 4000   # validation begins right after the training split
VAL_END = 5000     # validation stops at row 5,000 (1,000 samples total)

if None in (TRAIN_END, VAL_START, VAL_END):
    raise ValueError('Set TRAIN_END, VAL_START, and VAL_END according to the instructions.')

# `select` keeps only the rows in the given index range, producing each split.
train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))
display(df.head())

# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [ ]:
# Initialize the tokenizer and its padding behavior.
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'  # small GPT-2 checkpoint (try 'gpt2-medium'/'gpt2-large' for more capacity)
if MODEL_NAME is None:
    raise ValueError("Set MODEL_NAME to the pretrained checkpoint (e.g., 'gpt2').")

# Download the matching BPE tokenizer for the chosen checkpoint.
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
# GPT-2 has no dedicated padding token, so reuse the end-of-sequence token for padding.
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Configure how each SMS message is converted into fixed-length token sequences.
TEXT_COLUMN = 'sms'          # column holding the raw message text
PADDING_STRATEGY = 'max_length'  # pad every sequence up to MAX_SEQ_LEN
TRUNCATION_FLAG = True       # cut off any sequence longer than MAX_SEQ_LEN
MAX_SEQ_LEN = 64            # SMS messages are short, so 64 tokens is plenty

for setting in (TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, MAX_SEQ_LEN):
    if setting is None:
        raise ValueError('Complete TEXT_COLUMN, PADDING_STRATEGY, TRUNCATION_FLAG, and MAX_SEQ_LEN.')


# Tokenize a batch of examples into input_ids / attention_mask tensors.
def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )


# Apply the tokenizer to both splits; batched=True processes many rows at once for speed.
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [ ]:
# Instantiate GPT-2 with a sequence-classification head on top.
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2  # binary task: spam vs. ham
if NUM_LABELS is None:
    raise ValueError('Set NUM_LABELS to 2 for binary classification.')

model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    # The classification head reads the last non-padding token, so the model must
    # know which id is padding (the EOS id we reused above) to avoid errors.
    pad_token_id=tokenizer.eos_token_id,
)

# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [ ]:
# Implement scaled dot-product attention from scratch.
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # 1/sqrt(d_k): scaling keeps the dot products from growing with embed_dim,
        # which would otherwise push softmax into tiny-gradient (saturated) regions.
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        # Similarity of every query against every key: (batch, seq, seq).
        # key.transpose(-2, -1) swaps the last two dims so the matmul lines up.
        scores = torch.matmul(
            query,
            key.transpose(-2, -1),
        ) * self.scale
        if mask is not None:
            # Set masked positions to -inf so they receive ~0 weight after softmax.
            scores = scores.masked_fill(mask == 0, float('-inf'))
        # Normalize scores over the key dimension (last axis) into a probability distribution.
        attn = F.softmax(scores, dim=-1)
        # Weighted sum of the value vectors gives the attention output.
        return torch.matmul(attn, value), attn


class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        # Look up a learned vector for each token id.
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Single self-attention block over the embedded sequence.
        self.attn = Attention(embed_dim)
        # Linear classifier mapping the pooled representation to class logits.
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)                      # (batch, seq, embed_dim)
        # Self-attention: query, key and value are all the same embedded sequence.
        attn_output, _ = self.attn(embed, embed, embed)
        # Average over the sequence dimension to get one vector per message.
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)                          # (batch, num_classes)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [ ]:
# Preprocess datasets for the custom attention model.
ATTN_TEXT_COLUMN = 'sms'  # same text column as before
ATTN_MAX_LEN = 64         # same 64-token cap so both models see comparable context
if ATTN_TEXT_COLUMN is None or ATTN_MAX_LEN is None:
    raise ValueError('Complete ATTN_TEXT_COLUMN and ATTN_MAX_LEN.')


# Encode one example into a fixed-length list of token ids plus its label.
def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}


# Map over each split (not batched here, so we process one example at a time).
train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

In [ ]:
# Wrap the tokenized splits in a PyTorch Dataset so a DataLoader can batch them.
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Convert the stored python lists/ints into the tensors the model expects.
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }


TRAIN_DATA_FOR_LOADER = train_ds_attn  # tokenized training split
VAL_DATA_FOR_LOADER = val_ds_attn      # tokenized validation split
if TRAIN_DATA_FOR_LOADER is None or VAL_DATA_FOR_LOADER is None:
    raise ValueError('Assign TRAIN_DATA_FOR_LOADER and VAL_DATA_FOR_LOADER before creating loaders.')


# shuffle=True for training improves generalization; validation stays ordered.
train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER), batch_size=32)

In [ ]:
# Train the custom attention classifier.
vocab_size = len(tokenizer)  # full vocab size, including any tokens we added (pad/eos)
embed_dim = 64
num_classes = 2       # spam vs. ham
learning_rate = 1e-3  # typical starting LR for Adam on a small model
if None in (vocab_size, num_classes, learning_rate):
    raise ValueError('Set vocab_size, num_classes, and learning_rate before training.')


# Use the GPU when available, otherwise fall back to CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()  # standard loss for multi-class classification

# Train for a few epochs so the scratch model has time to actually learn the task.
EPOCHS = 3
attn_model.train()
for epoch in range(EPOCHS):
    for batch in train_loader:
        inputs = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()        # clear gradients from the previous step
        outputs = attn_model(inputs)  # forward pass -> class logits
        loss = criterion(outputs, labels)
        loss.backward()              # backprop the loss
        optimizer.step()             # update the weights
    print(f'Epoch {epoch + 1}/{EPOCHS} - last batch loss: {loss.item():.4f}')

print('Custom Attention model trained on SMS dataset. Sample batch loss:', loss.item())

# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [ ]:
# Configure evaluation metrics from the `evaluate` library.
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')    # fraction of correct predictions
precision = evaluate.load('precision')  # of predicted spam, how much is really spam
recall = evaluate.load('recall')        # of actual spam, how much we caught
f1 = evaluate.load('f1')                # harmonic mean of precision and recall


# Shared helper: turn raw logits + labels into the four metrics.
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)  # pick the highest-scoring class per example
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall': recall.compute(predictions=preds, references=labels)['recall'],
        'f1': f1.compute(predictions=preds, references=labels)['f1'],
    }

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [ ]:
# Evaluate GPT-2 on the validation split, one message at a time.
gpt2_preds = []
gpt2_labels = []
model.eval()  # disable dropout etc. for deterministic inference
for ex in val_tok:
    # Add a batch dimension and move the input to the model's device.
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    # Pass the attention_mask too: our sequences are padded with the EOS token, and
    # without the mask GPT-2 cannot tell padding from real tokens, which both triggers
    # a warning and can corrupt the prediction (it would attend to padding positions).
    attention_mask = torch.tensor(ex['attention_mask']).unsqueeze(0).to(model.device)
    with torch.no_grad():  # no gradients needed at inference -> saves memory/time
        logits = model(inputs, attention_mask=attention_mask).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()  # predicted class id
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])


# Compare predictions against the true labels with all four metrics.
gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)

In [ ]:
# Evaluate the custom attention model on the validation loader.
attn_preds = []
attn_labels = []
attn_model.eval()  # switch to inference mode
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)            # class logits for the batch
        preds = torch.argmax(outputs, dim=1)    # predicted class per example
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())


# Same four metrics so results line up directly with GPT-2's.
attn_metrics = {
    'accuracy': accuracy.compute(predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall': recall.compute(predictions=attn_preds, references=attn_labels)['recall'],
    'f1': f1.compute(predictions=attn_preds, references=attn_labels)['f1'],
}
print('Attention Model Metrics:', attn_metrics)

# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?

Attention works like a soft, content-based lookup table:

- **Query (Q):** represents the current token *asking* for information — "what am I looking for?"
- **Key (K):** represents what each token *offers* — an index used to decide how relevant that token is to the query.
- **Value (V):** the actual *content* (information) carried by each token that gets passed along once relevance is decided.

The model computes a similarity score between each query and every key (a dot product). Those scores are turned into weights with softmax, and the output is a **weighted sum of the values**. So Q×K decides *how much attention* to pay to each token, and V provides *what* is actually aggregated. In self-attention, Q, K and V are all derived from the same input sequence.

### 2. Why do we use a scaling factor in the dot-product attention?

The raw scores are dot products of query and key vectors of dimension $d_k$. As $d_k$ grows, the
variance of those dot products grows with it, so the scores become large in magnitude. Feeding
large values into softmax pushes it into a **saturated region** where it outputs values very close
to 0 and 1, and its **gradients become extremely small** (vanishing gradients), which makes
training slow and unstable.

Dividing the scores by $\sqrt{d_k}$ normalizes their variance back to roughly 1, keeping the
softmax in a sensitive range. This produces smoother attention distributions and healthier
gradients, which is why it is called *scaled* dot-product attention.

### 3. How does self-attention differ from traditional sequence models like RNNs?

| Aspect | RNN / LSTM | Self-Attention |
|---|---|---|
| **Processing** | Sequential — one token after another, hidden state carried forward | Parallel — all tokens processed at once |
| **Long-range dependencies** | Information must travel step-by-step; distant tokens are hard to connect and signals can vanish | Any token can directly attend to any other token in a single step, regardless of distance |
| **Path length between tokens** | O(sequence length) | O(1) (constant) |
| **Efficiency / hardware** | Recurrence prevents parallelization; slow on long sequences | Fully parallelizable, maps well onto GPUs |
| **Order awareness** | Built in through recurrence | Order-agnostic by itself, so positional encodings are added |

In short, RNNs read text **left-to-right with a memory bottleneck**, while self-attention compares
**every token to every other token simultaneously**, capturing long-range relationships more
directly and training much faster — at the cost of $O(n^2)$ compute in the sequence length.

### 4. Performance analysis

**Which model performed better?** Compare the printed `gpt2_metrics` and `attn_metrics`
dictionaries (accuracy, precision, recall, F1). In general, **GPT-2 has the advantage** because it
brings massive pretrained language knowledge, a deep multi-head/multi-layer Transformer, and
positional information — so even with a simple classification head it tends to reach high accuracy
and F1. The custom model is a single attention layer trained from scratch on only 4,000 messages,
so it usually trails GPT-2, though on this easy, well-separated SMS task it can still reach
respectable accuracy.

**Trade-offs:**
- *GPT-2:* higher accuracy and richer language understanding, but ~124M+ parameters — slow,
  memory-hungry, and overkill for such a short-text binary task.
- *Custom attention model:* tiny, fast, and easy to train/deploy, but it has no pretraining, no
  positional encoding, only one attention head/layer, and mean-pooling that discards word order —
  so it captures far less nuance and is weaker on subtle or rare spam patterns.

**One improvement for the custom classifier:** add **positional encodings** so the model is aware
of word order (currently mean-pooling makes it a bag-of-embeddings). Other strong options:
use **multi-head attention**, **stack several attention layers** with residual connections and
layer normalization, apply an **attention/padding mask** so padded tokens are ignored, or train
for **more epochs with a learning-rate schedule**.